# Holistic Data Preparer — Final Project
**Fintech Customer Credit Risk — End-to-End Data Preprocessing & Feature Engineering**

Role: Junior Data Scientist | Dataset: Customer Credit Risk (CSV + JSON + SQL + API)
Goal: produce a clean, fully-preprocessed dataset ready for an ML model predicting `default_flag`.


---
## Part A: Conceptual Foundation

### What is Data Analysis?
Data Analysis is the process of inspecting, cleaning, transforming, and modeling data to discover
useful information, draw conclusions, and support decision-making. It combines statistics, domain
knowledge, and tooling (Pandas, SQL, visualization) to turn raw data into actionable insight.

### How to Plan a Data Science Project
1. **Business understanding** — define the problem and success metric (here: predict loan default).
2. **Data acquisition** — gather data from all relevant sources (CSV, JSON, SQL, APIs).
3. **Data understanding** — explore structure, types, distributions, missingness.
4. **Data preparation** — clean, impute, treat outliers, encode, scale, engineer features.
5. **Modeling** — train/evaluate candidate ML models (out of scope for this project).
6. **Evaluation & deployment** — validate against business goals, deploy, monitor.

### How to Frame a Machine Learning Problem
Framing an ML problem means translating a business question into a well-posed learning task:
- **Task type**: classification (default vs. no default) — a supervised, binary classification problem.
- **Target variable**: `default_flag` (0 = No Default, 1 = Default).
- **Features**: demographic, financial, and behavioral attributes.
- **Evaluation metric**: e.g. ROC-AUC / F1 (imbalanced-risk classification), chosen based on the cost
  of false negatives (missed defaulters) vs. false positives.
- **Constraints**: data leakage must be avoided (e.g. don't use post-default information), and the
  data must be representative of the population the model will serve.

### Tensors — In-Depth Explanation (with NumPy)
A **tensor** is a generalized, n-dimensional array — the core data structure of numerical computing
and deep learning frameworks (NumPy, TensorFlow, PyTorch).

| Rank | Name    | Example              | NumPy shape |
|------|---------|-----------------------|-------------|
| 0    | Scalar  | `5`                   | `()`        |
| 1    | Vector  | `[1, 2, 3]`           | `(3,)`      |
| 2    | Matrix  | `[[1,2],[3,4]]`       | `(2,2)`     |
| 3+   | Tensor  | stack of matrices     | `(n,2,2)`   |

A tabular dataset like ours is naturally a **rank-2 tensor** (rows × columns).


In [4]:
import numpy as np

# Rank-0 tensor (scalar)
scalar = np.array(42)
print("Scalar:", scalar, "| shape:", scalar.shape, "| ndim:", scalar.ndim)

# Rank-1 tensor (vector) - e.g. one customer's numeric features
vector = np.array([38, 435760.0, 98946.8, 647])
print("\nVector:", vector, "| shape:", vector.shape, "| ndim:", vector.ndim)

# Rank-2 tensor (matrix) - e.g. a mini-batch of customers x features
matrix = np.array([
    [38, 435760.0, 98946.8, 647],
    [45, 612000.0, 150000.0, 710],
    [29, 288000.0, 60000.0, 590]
])
print("\nMatrix shape:", matrix.shape, "| ndim:", matrix.ndim)

# Rank-3 tensor - e.g. stacking multiple such batches over time (a "cube" of data)
tensor_3d = np.stack([matrix, matrix * 1.02, matrix * 0.98])
print("\n3D Tensor shape:", tensor_3d.shape, "| ndim:", tensor_3d.ndim)

# Common tensor operations
print("\nTranspose of matrix:\n", matrix.T)
print("\nElement-wise scaling (matrix * 2) first row:", (matrix * 2)[0])
print("\nMatrix mean along axis=0 (per-feature mean):", matrix.mean(axis=0))
print("Matrix mean along axis=1 (per-customer mean):", matrix.mean(axis=1))


Scalar: 42 | shape: () | ndim: 0

Vector: [3.80000e+01 4.35760e+05 9.89468e+04 6.47000e+02] | shape: (4,) | ndim: 1

Matrix shape: (3, 4) | ndim: 2

3D Tensor shape: (3, 3, 4) | ndim: 3

Transpose of matrix:
 [[3.80000e+01 4.50000e+01 2.90000e+01]
 [4.35760e+05 6.12000e+05 2.88000e+05]
 [9.89468e+04 1.50000e+05 6.00000e+04]
 [6.47000e+02 7.10000e+02 5.90000e+02]]

Element-wise scaling (matrix * 2) first row: [7.600000e+01 8.715200e+05 1.978936e+05 1.294000e+03]

Matrix mean along axis=0 (per-feature mean): [3.73333333e+01 4.45253333e+05 1.02982267e+05 6.49000000e+02]
Matrix mean along axis=1 (per-customer mean): [133847.95 190688.75  87154.75]


---
## Part B: Data Acquisition

Loading the Customer Credit Risk dataset from **four different sources**, exactly as a real
fintech data pipeline would:
- `transactions.csv` — main financial/behavioral dataset
- `customer_metadata.json` — demographic metadata
- `loan_repayment.db` (SQLite) — loan repayment history
- `economic_indicators_api.json` — dummy API response with external economic indicators


In [5]:
import json
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    OrdinalEncoder, LabelEncoder, OneHotEncoder,
    StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler,
    KBinsDiscretizer, FunctionTransformer, PowerTransformer
)
from sklearn.compose import ColumnTransformer

DATA_DIR = "/Users/devjoshi/Desktop/Data Preprocessig/Final_Project"   # adjust path if running from a different working directory
pd.set_option("display.max_columns", 50)


In [6]:
# 1. Load CSV (main transactions dataset)
df_csv = pd.read_csv(f"{DATA_DIR}/transactions.csv")
print("CSV loaded:", df_csv.shape)
df_csv.head()


CSV loaded: (1200, 10)


,customer_id,age,annual_income,loan_amount,loan_purpose,credit_score,transaction_count,spending_ratio,join_date,default_flag
0,CUST00001,60.0,261459.87,147692.98,Education,633.0,44,14.08,2015-04-15,0
1,CUST00002,44.0,393587.92,70387.07,Other,672.0,48,12.23,2017-09-24,0
2,CUST00003,19.0,767271.71,77355.93,Other,598.0,54,25.96,2023-12-13,1
3,CUST00004,18.0,340998.75,74790.99,Car,801.0,32,30.96,2024-10-26,0
4,CUST00005,NaN,286282.40,207168.38,Business,663.0,42,21.69,2020-04-02,1


In [9]:
# 2. Parse JSON (customer metadata)
with open(f"{DATA_DIR}/customer_metadata.json") as f:
    json_records = json.load(f)
df_json = pd.DataFrame(json_records)
print("JSON loaded:", df_json.shape)
df_json.head()


JSON loaded: (1200, 5)


,customer_id,gender,region,education_level,employment_type
0,CUST00001,Female,West,Graduate,Salaried
1,CUST00002,NaN,West,Graduate,Self-Employed
2,CUST00003,Female,North,Graduate,Salaried
3,CUST00004,Female,North,Graduate,Salaried
4,CUST00005,Male,North,Graduate,Salaried


In [10]:
# 3. Fetch records from SQL (loan repayment history)
conn = sqlite3.connect(f"{DATA_DIR}/loan_repayment.db")
df_sql = pd.read_sql("SELECT * FROM repayment_history", conn)
conn.close()
print("SQL loaded:", df_sql.shape)
df_sql.head()


SQL loaded: (1200, 2)


,customer_id,repayment_history
0,CUST00001,0
1,CUST00002,1
2,CUST00003,1
3,CUST00004,1
4,CUST00005,2


In [11]:
# 4. Fetch data from a dummy API (external economic indicators)
with open(f"{DATA_DIR}/economic_indicators_api.json") as f:
    api_response = json.load(f)
df_api = pd.DataFrame(api_response["data"])
print("API loaded:", df_api.shape)
df_api


API loaded: (4, 4)


,region,avg_interest_rate,inflation_rate,unemployment_rate
0,North,8.75,5.2,6.1
1,South,8.40,4.8,5.4
2,East,9.10,5.6,7.2
3,West,8.55,5.0,5.9


In [12]:
# Merge all 4 sources into one working dataframe on customer_id / region
df = df_csv.merge(df_json, on="customer_id", how="left") \
           .merge(df_sql, on="customer_id", how="left") \
           .merge(df_api, on="region", how="left")

df["join_date"] = pd.to_datetime(df["join_date"])
print("Merged dataset shape:", df.shape)
df.head()


Merged dataset shape: (1200, 18)


,customer_id,age,annual_income,loan_amount,loan_purpose,credit_score,transaction_count,spending_ratio,join_date,default_flag,gender,region,education_level,employment_type,repayment_history,avg_interest_rate,inflation_rate,unemployment_rate
0,CUST00001,60.0,261459.87,147692.98,Education,633.0,44,14.08,2015-04-15,0,Female,West,Graduate,Salaried,0,8.55,5.0,5.9
1,CUST00002,44.0,393587.92,70387.07,Other,672.0,48,12.23,2017-09-24,0,NaN,West,Graduate,Self-Employed,1,8.55,5.0,5.9
2,CUST00003,19.0,767271.71,77355.93,Other,598.0,54,25.96,2023-12-13,1,Female,North,Graduate,Salaried,1,8.75,5.2,6.1
3,CUST00004,18.0,340998.75,74790.99,Car,801.0,32,30.96,2024-10-26,0,Female,North,Graduate,Salaried,1,8.75,5.2,6.1
4,CUST00005,NaN,286282.40,207168.38,Business,663.0,42,21.69,2020-04-02,1,Male,North,Graduate,Salaried,2,8.75,5.2,6.1


---
## Part C: Data Understanding & Cleaning

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        1200 non-null   str           
 1   age                1128 non-null   float64       
 2   annual_income      1140 non-null   float64       
 3   loan_amount        1200 non-null   float64       
 4   loan_purpose       1200 non-null   str           
 5   credit_score       1152 non-null   float64       
 6   transaction_count  1200 non-null   int64         
 7   spending_ratio     1200 non-null   float64       
 8   join_date          1200 non-null   datetime64[us]
 9   default_flag       1200 non-null   int64         
 10  gender             1152 non-null   str           
 11  region             1200 non-null   str           
 12  education_level    1200 non-null   str           
 13  employment_type    1140 non-null   str           
 14  repayment_history  

In [14]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
customer_id,1200,1200,CUST00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,1128.0,NaN,NaN,NaN,38.41578,18.0,31.0,38.0,46.0,73.0,10.827518
annual_income,1140.0,NaN,NaN,NaN,596252.193675,95661.0,322348.13,435760.035,646384.37,15806205.12,792295.919287
loan_amount,1200.0,NaN,NaN,NaN,129353.782058,16158.01,67585.5675,98946.77,148011.2075,1761156.22,131655.597709
loan_purpose,1200,5,Home,358,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credit_score,1152.0,NaN,NaN,NaN,648.779514,200.0,595.0,647.0,704.0,950.0,88.965656
transaction_count,1200.0,NaN,NaN,NaN,45.3525,26.0,41.0,45.0,50.0,69.0,6.572613
spending_ratio,1200.0,NaN,NaN,NaN,30.888508,1.0,14.6075,25.59,41.1825,130.0,21.789579
join_date,1200,NaN,NaN,NaN,2020-09-30 15:18:00,2015-01-03 00:00:00,2017-11-08 12:00:00,2020-08-31 00:00:00,2023-08-05 06:00:00,2026-06-28 00:00:00,NaN
default_flag,1200.0,NaN,NaN,NaN,0.540833,0.0,0.0,1.0,1.0,1.0,0.498538


In [15]:
# Data-quality report (Pandas-Profiling style, done manually since ydata-profiling
# may not be installed in every environment)
missing_summary = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
quality_report = pd.DataFrame({"missing_count": missing_summary, "missing_pct": missing_pct})
quality_report = quality_report[quality_report.missing_count > 0].sort_values("missing_count", ascending=False)
print("=== Data Quality Report: Missing Values ===")
quality_report


=== Data Quality Report: Missing Values ===


,missing_count,missing_pct
age,72,6.0
annual_income,60,5.0
employment_type,60,5.0
credit_score,48,4.0
gender,48,4.0


*(Optional) To generate a full interactive HTML profiling report, install `ydata-profiling`
and run:*
```python
from ydata_profiling import ProfileReport
ProfileReport(df, title="Customer Credit Risk - Profiling Report").to_file("profiling_report.html")
```


In [16]:
# --- Missing value handling: multiple strategies, matched to the column that suits it ---

# 1) Simple Imputer (numerical: mean) -> age
num_imputer_mean = SimpleImputer(strategy="mean")
df["age"] = num_imputer_mean.fit_transform(df[["age"]])

# 2) Simple Imputer (categorical: most frequent) -> gender & employment_type
cat_imputer_freq = SimpleImputer(strategy="most_frequent")
df["gender"] = cat_imputer_freq.fit_transform(df[["gender"]]).ravel()
df["employment_type"] = cat_imputer_freq.fit_transform(df[["employment_type"]]).ravel()

# 3) Missing Indicator + Random Sample Imputation -> annual_income
df["annual_income_was_missing"] = df["annual_income"].isna().astype(int)
observed = df["annual_income"].dropna()
missing_mask = df["annual_income"].isna()
df.loc[missing_mask, "annual_income"] = np.random.default_rng(1).choice(
    observed.values, size=missing_mask.sum(), replace=True
)

# 4) KNN Imputer (multivariate) -> credit_score, using correlated numeric columns
knn_cols = ["credit_score", "annual_income", "loan_amount", "age"]
knn_imputer = KNNImputer(n_neighbors=5)
df[knn_cols] = knn_imputer.fit_transform(df[knn_cols])

# NOTE on MICE: sklearn's IterativeImputer implements the MICE algorithm.
# from sklearn.experimental import enable_iterative_imputer
# from sklearn.impute import IterativeImputer
# mice_imputer = IterativeImputer(random_state=0)
# df[knn_cols] = mice_imputer.fit_transform(df[knn_cols])   # alternative to KNNImputer above

print("Missing values remaining after imputation:")
remaining = df.isna().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else "None - dataset fully imputed.")


Missing values remaining after imputation:
None - dataset fully imputed.


---
## Part D: Outlier Handling

Detecting outliers with **Z-score**, **IQR**, and **Percentile** methods, then treating them with
**Winsorization** (capping) on the numeric columns flagged as having outliers in the brief:
`annual_income`, `loan_amount`, `credit_score`.


In [17]:
def zscore_outliers(series, thresh=3):
    z = np.abs(stats.zscore(series))
    return z > thresh

def iqr_outliers(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - k * iqr) | (series > q3 + k * iqr)

def percentile_outliers(series, lower=0.01, upper=0.99):
    lo, hi = series.quantile(lower), series.quantile(upper)
    return (series < lo) | (series > hi)

def winsorize(series, lower=0.01, upper=0.99):
    lo, hi = series.quantile(lower), series.quantile(upper)
    return series.clip(lo, hi)

for col in ["annual_income", "loan_amount", "credit_score"]:
    z_flag, iqr_flag, pct_flag = zscore_outliers(df[col]), iqr_outliers(df[col]), percentile_outliers(df[col])
    print(f"{col:15s} -> Z-score: {z_flag.sum():4d} | IQR: {iqr_flag.sum():4d} | Percentile: {pct_flag.sum():4d}")


annual_income   -> Z-score:   20 | IQR:   62 | Percentile:   24
loan_amount     -> Z-score:   21 | IQR:   67 | Percentile:   24
credit_score    -> Z-score:   12 | IQR:   24 | Percentile:   19


In [18]:
# Treat outliers via Winsorization (cap at 1st/99th percentile)
for col in ["annual_income", "loan_amount", "credit_score"]:
    df[col] = winsorize(df[col])

# credit_score has a hard domain of 300-850 regardless of percentile capping
df["credit_score"] = df["credit_score"].clip(300, 850)

print("Outliers capped via Winsorization on: annual_income, loan_amount, credit_score")
df[["annual_income", "loan_amount", "credit_score"]].describe()


Outliers capped via Winsorization on: annual_income, loan_amount, credit_score


,annual_income,loan_amount,credit_score
count,1.200000e+03,1200.000000,1200.000000
mean,5.697487e+05,125083.797954,648.540067
std,5.153325e+05,97630.631202,81.230103
min,1.371915e+05,23557.876300,454.990000
25%,3.234763e+05,67585.567500,596.000000
50%,4.358040e+05,98946.770000,646.000000
75%,6.476025e+05,148011.207500,702.000000
max,4.053345e+06,647924.861600,850.000000


---
## Part E: Feature Engineering — Variable Types, Dates & Encoding

In [19]:
# --- Handling mixed & date variables ---
# gender = categorical, age = numeric, spending_ratio = numeric (skewed)
df["join_year"] = df["join_date"].dt.year
df["join_month"] = df["join_date"].dt.month
df["join_day"] = df["join_date"].dt.day
df["join_weekday"] = df["join_date"].dt.day_name()
df[["join_date", "join_year", "join_month", "join_day", "join_weekday"]].head()


,join_date,join_year,join_month,join_day,join_weekday
0,2015-04-15,2015,4,15,Wednesday
1,2017-09-24,2017,9,24,Sunday
2,2023-12-13,2023,12,13,Wednesday
3,2024-10-26,2024,10,26,Saturday
4,2020-04-02,2020,4,2,Thursday


In [20]:
# --- Ordinal Encoding -> education_level ---
edu_order = [["Primary", "Secondary", "Graduate", "Post-Graduate"]]
ordinal_enc = OrdinalEncoder(categories=edu_order)
df["education_level_encoded"] = ordinal_enc.fit_transform(df[["education_level"]])

# --- Label Encoding -> gender ---
label_enc = LabelEncoder()
df["gender_encoded"] = label_enc.fit_transform(df["gender"])
print("Label mapping (gender):", dict(zip(label_enc.classes_, range(len(label_enc.classes_)))))

# --- One-Hot Encoding -> region, loan_purpose ---
df = pd.get_dummies(df, columns=["region", "loan_purpose"], prefix=["region", "purpose"])

df[[c for c in df.columns if "region_" in c or "purpose_" in c or "encoded" in c]].head()


Label mapping (gender): {'Female': 0, 'Male': 1, 'Other': 2}


,education_level_encoded,gender_encoded,region_East,region_North,region_South,region_West,purpose_Business,purpose_Car,purpose_Education,purpose_Home,purpose_Other
0,2.0,0,False,False,False,True,False,False,True,False,False
1,2.0,1,False,False,False,True,False,False,False,False,True
2,2.0,0,False,True,False,False,False,False,False,False,True
3,2.0,0,False,True,False,False,False,True,False,False,False
4,2.0,1,False,True,False,False,True,False,False,False,False


In [21]:
# --- Numerical encoding: repayment_history & transaction_count already numeric ---
df["repayment_history"] = df["repayment_history"].astype(int)
df["transaction_count"] = df["transaction_count"].astype(int)

# --- Binning / Discretization -> annual_income into quantile bands ---
df["income_band"] = pd.qcut(df["annual_income"], q=4, labels=["Low", "Mid-Low", "Mid-High", "High"])

# --- Binarization -> flag credit_score > 700 ---
df["good_credit_flag"] = (df["credit_score"] > 700).astype(int)

# --- Quantile Binning -> repayment_history ---
df["repayment_history_qbin"] = pd.qcut(df["repayment_history"].rank(method="first"), q=3,
                                        labels=["Low_Missed", "Med_Missed", "High_Missed"])

# --- K-Means Binning -> transaction_count ---
kbins = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="kmeans")
df["transaction_count_kmeans_bin"] = kbins.fit_transform(df[["transaction_count"]]).astype(int)

df[["income_band", "good_credit_flag", "repayment_history_qbin", "transaction_count_kmeans_bin"]].head()


,income_band,good_credit_flag,repayment_history_qbin,transaction_count_kmeans_bin
0,Low,0,Low_Missed,1
1,Mid-Low,0,Low_Missed,2
2,High,0,Low_Missed,2
3,Mid-Low,1,Low_Missed,0
4,Low,0,High_Missed,1


---
## Part F: Feature Scaling

Applying **Standardization**, **Normalization/MinMax**, **MaxAbs**, and **Robust** scaling.


In [22]:
scale_cols = ["annual_income", "loan_amount"]
all_numeric = ["age", "annual_income", "loan_amount", "credit_score",
               "repayment_history", "transaction_count", "spending_ratio"]

# Standardization (Z-score scaling)
std_scaler = StandardScaler()
df[[c + "_standard" for c in scale_cols]] = std_scaler.fit_transform(df[scale_cols])

# Min-Max Scaling (Normalization)
minmax_scaler = MinMaxScaler()
df[[c + "_minmax" for c in all_numeric]] = minmax_scaler.fit_transform(df[all_numeric])

# MaxAbs Scaling
maxabs_scaler = MaxAbsScaler()
df[[c + "_maxabs" for c in all_numeric]] = maxabs_scaler.fit_transform(df[all_numeric])

# Robust Scaling (median / IQR - robust to outliers)
robust_scaler = RobustScaler()
df[[c + "_robust" for c in all_numeric]] = robust_scaler.fit_transform(df[all_numeric])

print("Scaling complete.")
df[[c + "_standard" for c in scale_cols] + [c + "_minmax" for c in all_numeric[:3]]].describe()


Scaling complete.


,annual_income_standard,loan_amount_standard,age_minmax,annual_income_minmax,loan_amount_minmax
count,1.200000e+03,1.200000e+03,1200.000000,1200.000000,1200.000000
mean,-1.924387e-17,2.190840e-16,0.371196,0.110455,0.162606
std,1.000417e+00,1.000417e+00,0.190862,0.131591,0.156367
min,-8.397249e-01,-1.040332e+00,0.000000,0.000000,0.000000
25%,-4.780894e-01,-5.891819e-01,0.236364,0.047568,0.070516
50%,-2.600273e-01,-2.678250e-01,0.371196,0.076251,0.120745
75%,1.511378e-01,2.349362e-01,0.490909,0.130335,0.199327
max,6.762719e+00,5.357530e+00,1.000000,1.000000,1.000000


---
## Part G: Feature Construction & Transformation

In [23]:
# --- FunctionTransformer: log, reciprocal, sqrt on spending_ratio ---
df["spending_ratio_log"] = FunctionTransformer(np.log1p).transform(df[["spending_ratio"]])
df["spending_ratio_sqrt"] = FunctionTransformer(np.sqrt).transform(df[["spending_ratio"]])
df["spending_ratio_reciprocal"] = FunctionTransformer(lambda x: 1 / (x + 1)).transform(df[["spending_ratio"]])

# --- PowerTransformer: Box-Cox (loan_amount) and Yeo-Johnson (annual_income) ---
pt_boxcox = PowerTransformer(method="box-cox")
df["loan_amount_boxcox"] = pt_boxcox.fit_transform(df[["loan_amount"]].clip(lower=1))

pt_yeo = PowerTransformer(method="yeo-johnson")
df["annual_income_yeojohnson"] = pt_yeo.fit_transform(df[["annual_income"]])

print("Skew before -> after Box-Cox (loan_amount):",
      round(df["loan_amount"].skew(), 3), "->", round(df["loan_amount_boxcox"].skew(), 3))
print("Skew before -> after Yeo-Johnson (annual_income):",
      round(df["annual_income"].skew(), 3), "->", round(df["annual_income_yeojohnson"].skew(), 3))


Skew before -> after Box-Cox (loan_amount): 2.912 -> -0.004
Skew before -> after Yeo-Johnson (annual_income): 4.621 -> -0.024


In [24]:
# --- New feature construction ---
df["debt_to_income_ratio"] = (df["loan_amount"] / df["annual_income"]).round(4)
df["avg_monthly_transactions"] = (df["transaction_count"] / 6).round(2)
df["spending_to_income_constructed"] = ((df["spending_ratio"] / 100)).round(4)

df[["debt_to_income_ratio", "avg_monthly_transactions", "spending_to_income_constructed"]].describe()


,debt_to_income_ratio,avg_monthly_transactions,spending_to_income_constructed
count,1200.000000,1200.000000,1200.000000
mean,0.310118,7.558683,0.308885
std,0.328750,1.095529,0.217896
min,0.009900,4.330000,0.010000
25%,0.127425,6.830000,0.146075
50%,0.223650,7.500000,0.255900
75%,0.363900,8.330000,0.411825
max,3.403900,11.500000,1.300000


In [25]:
# --- ColumnTransformer: apply different preprocessing to categorical vs numeric
#     features in ONE pipeline (useful for a downstream ML Pipeline / model) ---
numeric_features = ["age", "annual_income", "loan_amount", "credit_score"]
categorical_features = ["employment_type"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])
ct_output = preprocessor.fit_transform(df)
print("ColumnTransformer output shape:", ct_output.shape,
      "(scaled numeric + one-hot categorical, in a single fit_transform call)")


ColumnTransformer output shape: (1200, 7) (scaled numeric + one-hot categorical, in a single fit_transform call)


---
## Part H: Final Deliverable

In [26]:
print("Final dataset shape:", df.shape)
print("Missing values remaining:", int(df.isna().sum().sum()))
print("Numeric columns:", df.select_dtypes(include=[np.number]).shape[1])
print("Categorical/object columns:", df.select_dtypes(include=["object", "category"]).shape[1])

df.to_csv("outputs/final_cleaned_dataset.csv", index=False)
print("\nSaved -> outputs/final_cleaned_dataset.csv")
df.head()


Final dataset shape: (1200, 67)
Missing values remaining: 0
Numeric columns: 50
Categorical/object columns: 7


/var/folders/2b/zfhcjjhs3x3d0_y11r9nxwnw0000gn/T/ipykernel_1190/3904173745.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print("Categorical/object columns:", df.select_dtypes(include=["object", "category"]).shape[1])


OSError: Cannot save file into a non-existent directory: 'outputs'

### Report Summary

- **Missing value strategies**: Mean imputation (`age`), most-frequent imputation (`gender`,
  `employment_type`), missing-indicator + random-sample imputation (`annual_income`), and a
  multivariate KNN imputer (`credit_score`, using `annual_income`, `loan_amount`, `age`). All
  strategies were matched to the nature and missingness pattern of each column; after imputation,
  0 missing values remain.
- **Outlier handling**: Z-score, IQR, and percentile methods were used to *detect* outliers in
  `annual_income`, `loan_amount`, and `credit_score`; Winsorization (1st/99th percentile capping)
  was used to *treat* them, preserving row count while limiting the influence of extreme values.
- **Encoding**: Ordinal (`education_level`), Label (`gender`), One-Hot (`region`, `loan_purpose`),
  plus Binning/Discretization, Binarization, Quantile Binning, and K-Means Binning were applied to
  numeric features to create model-friendly categorical/derived variables.
- **Scaling / transformations**: Standardization, Min-Max, MaxAbs, and Robust scaling were applied;
  Log/Sqrt/Reciprocal and Box-Cox/Yeo-Johnson transforms reduced skew in `spending_ratio`,
  `loan_amount`, and `annual_income`.
- **New engineered features**: `debt_to_income_ratio`, `avg_monthly_transactions`, date parts
  (`join_year/month/day/weekday`), and several binned/flag features.
- **Final dataset readiness**: 1200 rows, 0 missing values, numeric + encoded + scaled +
  transformed features all present — ready for ML modeling of `default_flag`.
